# Week 6 ChronoPDE — Kaggle training

This notebook runs the mandatory four-trajectory gate, then trains the boundary-aware DCT model for up to 150 epochs. It uses GPU 0, saves `last.pt` after every completed epoch, stops before Kaggle's session limit, and always packages resumable state.

Attach the private dataset containing `chronopde.h5`; optionally include a previous `chronopde-full-train-s0` checkpoint folder. Enable a T4 GPU and Internet, then use **Save Version → Save & Run All**.

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

import hashlib
import json
import shutil
import signal
import subprocess
import sys
import time
import zipfile
from datetime import datetime
from pathlib import Path

SESSION_STARTED = time.monotonic()
SAFE_WALL_TIME_HOURS = 10.5
IMPLEMENTATION_COMMIT = '47fb45711391b34f5d06e6e5dc3808f18ef60665'

import torch
print('PyTorch:', torch.__version__)
assert torch.cuda.is_available(), 'Enable GPU in Kaggle Notebook settings'
print('GPU:', torch.cuda.get_device_name(0))
print('Visible GPUs:', torch.cuda.device_count(), '(trainer intentionally uses GPU 0)')

## Verify the private dataset and discover resumable state

In [ ]:
KAGGLE_INPUT = Path('/kaggle/input')
EXPECTED_DATA_SIZE = 2_389_261_328
EXPECTED_DATA_SHA256 = '907aa0d79e604e68ce2d4f5cccfd93ffc64eb68c473caf3edae4be438472caec'

data_candidates = [p for p in KAGGLE_INPUT.rglob('chronopde.h5') if p.is_file() and p.stat().st_size == EXPECTED_DATA_SIZE]
assert data_candidates, 'Attach the private dataset containing chronopde.h5'
DATA = data_candidates[0]
digest = hashlib.sha256()
with DATA.open('rb') as stream:
    for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
        digest.update(block)
assert digest.hexdigest() == EXPECTED_DATA_SHA256, 'Dataset checksum mismatch'
print('Verified dataset:', DATA)

def checkpoint_epoch(path):
    payload = torch.load(path, map_location='cpu', weights_only=False)
    assert payload.get('model_name') == 'chronopde', f'Wrong model checkpoint: {path}'
    return int(payload['epoch']) + 1

IMPORTED_ZIPS = Path('/kaggle/working/imported_chronopde_checkpoints')
for archive in KAGGLE_INPUT.rglob('chronopde_week6_*.zip'):
    with zipfile.ZipFile(archive) as bundle:
        if any(name.endswith('chronopde-full-train-s0/last.pt') for name in bundle.namelist()):
            destination = IMPORTED_ZIPS / archive.stem
            destination.mkdir(parents=True, exist_ok=True)
            bundle.extractall(destination)

checkpoint_roots = (KAGGLE_INPUT, IMPORTED_ZIPS)
checkpoint_candidates = [p for root in checkpoint_roots if root.exists() for p in root.rglob('last.pt') if p.parent.name == 'chronopde-full-train-s0']
checkpoint_candidates.sort(key=checkpoint_epoch, reverse=True)
INPUT_RUN_DIRECTORY = checkpoint_candidates[0].parent if checkpoint_candidates else None
INPUT_EPOCHS = checkpoint_epoch(checkpoint_candidates[0]) if checkpoint_candidates else 0
print('Input checkpoint epochs:', INPUT_EPOCHS)

## Install the pinned Week 6 implementation

In [ ]:
REPOSITORY_URL = 'https://github.com/madhavkapoor13/ChronoPDE.git'
REPOSITORY = Path('/kaggle/working/Chrono_pde')
if not REPOSITORY.exists():
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(REPOSITORY)], check=True)
subprocess.run(['git', '-C', str(REPOSITORY), 'fetch', 'origin', IMPLEMENTATION_COMMIT], check=True)
subprocess.run(['git', '-C', str(REPOSITORY), 'checkout', '--detach', IMPLEMENTATION_COMMIT], check=True)
os.chdir(REPOSITORY)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--ignore-requires-python', '-e', '.'], check=True)
actual_commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
assert actual_commit == IMPLEMENTATION_COMMIT
print('Pinned commit:', actual_commit)

## Initialize writable state and run the smoke gate

In [ ]:
STATE_ROOT = Path('/kaggle/working/chronopde_state')
WRITABLE_RUNS = STATE_ROOT / 'runs'
OUTPUT_RUN = WRITABLE_RUNS / 'chronopde-full-train-s0'
LOCAL_RUNS = REPOSITORY / 'artifacts/runs'
if INPUT_RUN_DIRECTORY is not None:
    OUTPUT_RUN.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(INPUT_RUN_DIRECTORY, OUTPUT_RUN, dirs_exist_ok=True)
LOCAL_RUNS.parent.mkdir(parents=True, exist_ok=True)
if LOCAL_RUNS.is_symlink():
    pass
elif not LOCAL_RUNS.exists() or (LOCAL_RUNS.is_dir() and not any(LOCAL_RUNS.iterdir())):
    if LOCAL_RUNS.exists(): LOCAL_RUNS.rmdir()
    LOCAL_RUNS.symlink_to(WRITABLE_RUNS, target_is_directory=True)
else:
    raise RuntimeError(f'Cannot safely replace {LOCAL_RUNS}')

if INPUT_EPOCHS == 0:
    smoke = subprocess.run([sys.executable, 'scripts/train.py', '--config', 'configs/project.yaml', '--model', 'chronopde', '--regime', 'full', '--seed', '0', '--device', 'cuda', '--data-path', str(DATA), '--smoke-overfit'])
    smoke_summary = LOCAL_RUNS / 'chronopde-full-train-s0-smoke/summary.json'
    assert smoke.returncode == 0 and smoke_summary.is_file()
    assert json.loads(smoke_summary.read_text()).get('passed') is True, 'Smoke gate failed; stop and debug before full training'
    print('Four-trajectory gate passed')
else:
    print('Resuming an existing full run; smoke gate already completed')

## Train with epoch-level checkpoints and a safe cutoff

In [ ]:
METRICS = OUTPUT_RUN / 'metrics.jsonl'
SUMMARY = OUTPUT_RUN / 'summary.json'
POLL_SECONDS = 60

def read_metrics():
    if not METRICS.is_file() or not METRICS.stat().st_size:
        return [], None
    rows = [json.loads(line) for line in METRICS.read_text().splitlines() if line.strip()]
    return rows, rows[-1]

command = [sys.executable, 'scripts/train.py', '--config', 'configs/project.yaml', '--model', 'chronopde', '--regime', 'full', '--seed', '0', '--device', 'cuda', '--data-path', str(DATA), '--resume']
process = subprocess.Popen(command)
deadline = SESSION_STARTED + SAFE_WALL_TIME_HOURS * 3600
previous_count = -1
stopped_for_cutoff = False
while process.poll() is None:
    rows, latest = read_metrics()
    if len(rows) != previous_count:
        print(datetime.now().isoformat(timespec='seconds'), 'completed epochs:', len(rows), 'latest:', latest)
        previous_count = len(rows)
    if time.monotonic() >= deadline:
        process.send_signal(signal.SIGINT)
        stopped_for_cutoff = True
        break
    time.sleep(POLL_SECONDS)
if stopped_for_cutoff:
    try:
        process.wait(timeout=300)
    except subprocess.TimeoutExpired:
        process.terminate(); process.wait(timeout=60)
else:
    process.wait()
rows, latest = read_metrics()
print('Return code:', process.returncode, 'completed epochs:', len(rows), 'latest:', latest)
assert (OUTPUT_RUN / 'last.pt').is_file(), 'No resumable checkpoint was produced'

## Package state and conditionally run frozen ID evaluation

In [ ]:
from IPython.display import FileLink, display

checkpoint_zip = shutil.make_archive('/kaggle/working/chronopde_week6_checkpoint', 'zip', root_dir=WRITABLE_RUNS, base_dir='chronopde-full-train-s0')
print('Resumable package:', checkpoint_zip)
display(FileLink(checkpoint_zip))

passed = SUMMARY.is_file() and json.loads(SUMMARY.read_text()).get('passed') is True
if passed:
    best = LOCAL_RUNS / 'chronopde-full-train-s0/best.pt'
    subprocess.run([sys.executable, 'scripts/evaluate.py', '--config', 'configs/project.yaml', '--model', 'chronopde', '--experiment', 'id_rollout', '--checkpoint', str(best), '--device', 'cuda', '--data-path', str(DATA)], check=True)
    local_report = REPOSITORY / 'reports/baselines/week6/chronopde'
    output_report = STATE_ROOT / 'reports/baselines/week6/chronopde'
    output_report.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(local_report, output_report, dirs_exist_ok=True)
    print((output_report / 'summary.json').read_text())
else:
    print('Training is incomplete or the gate failed; frozen ID evaluation was skipped')

final_zip = shutil.make_archive('/kaggle/working/chronopde_week6_final', 'zip', root_dir=STATE_ROOT)
print('Final package:', final_zip)
display(FileLink(final_zip))